<a href="https://colab.research.google.com/github/averkina/MDEN-SOC-SOH/blob/main/SOC%26SOH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from __future__ import annotations

import os
import pickle
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

DATA_DIR = "/content/drive/MyDrive/"
Q_RATED = 1.1
SEQ_LEN = 32
PRED_HORIZON = 8
STRIDE = 8
K_TOP_FREQ = 3
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-3
NUM_WORKERS = 0
SEED = 42

TRAIN_CELLS = [f"b1c{i}" for i in range(14, 22)]
TEST_CELLS = ["b1c12", "b1c13"]


def set_seed(seed: int = SEED) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
# Dataset loading
def load_bat_dict(data_dir: str = DATA_DIR) -> dict:
    with open(os.path.join(data_dir, "batch1.pkl"), "rb") as f:
        batch1 = pickle.load(f)
    with open(os.path.join(data_dir, "batch2.pkl"), "rb") as f:
        batch2 = pickle.load(f)
    with open(os.path.join(data_dir, "batch3.pkl"), "rb") as f:
        batch3 = pickle.load(f)

    for k in ["b1c8", "b1c10", "b1c22"]:
        batch1.pop(k, None)
    for k in ["b3c37", "b3c2", "b3c23", "b3c32", "b3c42", "b3c43"]:
        batch3.pop(k, None)

    batch2_keys = ["b2c7", "b2c8", "b2c9", "b2c15", "b2c16"]
    batch1_keys = ["b1c0", "b1c1", "b1c2", "b1c3", "b1c4"]
    for i, bk in enumerate(batch1_keys):
        if bk in batch1 and batch2_keys[i] in batch2:
            last = len(batch1[bk]["cycles"])
            for j, jk in enumerate(batch2[batch2_keys[i]]["cycles"].keys()):
                batch1[bk]["cycles"][str(last + j)] = batch2[batch2_keys[i]]["cycles"][jk]
    for k in batch2_keys:
        batch2.pop(k, None)

    bat_dict = {**batch1, **batch2, **batch3}
    print(f"Loaded cells: {len(bat_dict)}")
    return bat_dict

In [ ]:
# Labels
def _align_cycle(cyc: dict) -> Optional[Tuple[np.ndarray, ...]]:
    I = np.asarray(cyc["I"], dtype=np.float64).ravel()
    V = np.asarray(cyc["V"], dtype=np.float64).ravel()
    T = np.asarray(cyc["T"], dtype=np.float64).ravel()
    t = np.asarray(cyc["t"], dtype=np.float64).ravel()
    n = min(len(I), len(V), len(T), len(t))
    if n < 10:
        return None
    Qd = None
    if "Qd" in cyc:
        Qd = np.asarray(cyc["Qd"], dtype=np.float64).ravel()[:n]
    return I[:n], V[:n], T[:n], t[:n], Qd


def _dt_hours(t: np.ndarray) -> np.ndarray:
    dt = np.diff(t, prepend=t[0])
    if len(dt) > 1:
        dt[0] = dt[1]
    dt = np.maximum(dt, 0.0)
    span = float(t[-1] - t[0]) if len(t) else 0.0
    med = float(np.median(dt)) if len(dt) else 0.0
    if span > 5.0 or med > 0.05:
        return dt / 60.0
    return dt


def compute_soc_soh(
    I: np.ndarray,
    t: np.ndarray,
    Qd: Optional[np.ndarray] = None,
    q_rated: float = Q_RATED,) -> Tuple[np.ndarray, float, float]:
    dt_h = _dt_hours(t)

    Qs = np.cumsum(I * dt_h)
    Qs = Qs - np.min(Qs)

    q_max = 0.0
    if Qd is not None and len(Qd) > 0 and np.any(np.isfinite(Qd)):
        q_max = float(np.nanmax(Qd))
    if q_max <= 1e-6:
        q_max = float(np.max(Qs)) if np.max(Qs) > 1e-6 else 1e-6

    soc = np.clip(Qs / q_max, 0.0, 1.0).astype(np.float32)
    soh = float(np.clip(q_max / q_rated, 0.0, 1.2))
    return soc, soh, q_max


def is_valid_soc_cycle(soc: np.ndarray, t: np.ndarray) -> bool:

    if len(soc) < 10 or len(t) < 10:
        return False
    if np.any(~np.isfinite(soc)) or np.any(~np.isfinite(t)):
        return False
    if (np.nanmax(soc) - np.nanmin(soc)) < 0.2:
        return False
    duration_min = (t[-1] - t[0])
    if duration_min > 200:
        duration_min = duration_min / 60.0
    if duration_min < 10 or duration_min > 100:
        return False
    if np.any(soc < -0.01) or np.any(soc > 1.01):
        return False
    dsoc = np.diff(soc)
    if len(dsoc) > 0 and float(np.std(dsoc)) > 0.05:
        return False
    return True


def build_cycle_series(bat_dict: dict) -> List[dict]:
    series_list: List[dict] = []
    prev_soh: Dict[str, float] = {}

    for bat_id, cell in bat_dict.items():
        try:
            cyc_items = sorted(cell["cycles"].items(), key=lambda kv: int(kv[0]))
        except ValueError:
            cyc_items = list(cell["cycles"].items())

        for cyc_id, cyc in cyc_items:
            aligned = _align_cycle(cyc)
            if aligned is None:
                continue
            I, V, T, t, Qd = aligned
            soc, soh, q_max = compute_soc_soh(I, t, Qd=Qd)
            if bat_id in prev_soh and abs(soh - prev_soh[bat_id]) > 0.02:
                prev_soh[bat_id] = soh
                continue
            prev_soh[bat_id] = soh

            if not is_valid_soc_cycle(soc, t):
                continue

            mask = I > 0
            if int(np.sum(mask)) < SEQ_LEN + PRED_HORIZON:
                continue

            try:
                cycle_idx = float(int(cyc_id))
            except ValueError:
                cycle_idx = float(len(series_list))

            t_c = t[mask]
            t_rel = (t_c - t_c[0]).astype(np.float32)

            series_list.append(
                {
                    "bat_id": bat_id,
                    "cycle_id": cyc_id,
                    "cycle": np.full(int(np.sum(mask)), cycle_idx, dtype=np.float32),
                    "time": t_rel,
                    "V": V[mask].astype(np.float32),
                    "I": I[mask].astype(np.float32),
                    "T": T[mask].astype(np.float32),
                    "SOC": soc[mask].astype(np.float32),
                    "SOH": np.full(int(np.sum(mask)), soh, dtype=np.float32),
                    "q_max": q_max,
                }
            )

    soh_vals = np.array([s["SOH"][0] for s in series_list], dtype=np.float64)
    print(f"Valid charge-cycle series: {len(series_list)}")
    if len(soh_vals):
        print(
            f"SOH labels — min {soh_vals.min():.3f}, max {soh_vals.max():.3f}, "
            f"mean {soh_vals.mean():.3f}, std {soh_vals.std():.3f}"
        )
    return series_list


def normalize_series(
    series_list: List[dict], stats: Optional[dict] = None
) -> Tuple[List[dict], dict]:
    keys = ["cycle", "time", "V", "I", "T"]
    if stats is None:
        cat = {k: np.concatenate([s[k] for s in series_list]) for k in keys}
        stats = {
            k: {
                "mean": float(np.mean(cat[k])),
                "std": float(max(np.std(cat[k]), 1e-6)),
            }
            for k in keys
        }
        print("Feature norm (train, reused for val/test):")
        for k in keys:
            print(f"  {k:5s}  mean={stats[k]['mean']:.4f}  std={stats[k]['std']:.4f}")

    out = []
    for s in series_list:
        ns = dict(s)
        for k in keys:
            ns[k] = ((s[k] - stats[k]["mean"]) / stats[k]["std"]).astype(np.float32)
        out.append(ns)
    return out, stats


class SOCSOHDataset(Dataset):
    def __init__(
        self,
        series_list: List[dict],
        cell_ids: Optional[List[str]] = None,
        seq_len: int = SEQ_LEN,
        pred_horizon: int = PRED_HORIZON,
        stride: int = STRIDE,
    ):
        self.samples: List[dict] = []
        total = seq_len + pred_horizon
        allowed = set(cell_ids) if cell_ids is not None else None

        for s in series_list:
            if allowed is not None and s["bat_id"] not in allowed:
                continue
            x_all = np.stack(
                [s["cycle"], s["time"], s["V"], s["I"], s["T"]], axis=-1
            ).astype(np.float32)
            soc = s["SOC"]
            soh = s["SOH"]
            L = len(soc)
            for start in range(0, L - total + 1, stride):
                end_in = start + seq_len
                end_out = end_in + pred_horizon
                self.samples.append(
                    {
                        "x": x_all[start:end_in].T.copy(),
                        "y_soc": soc[end_in:end_out].copy(),
                        "y_soh": soh[end_in:end_out].copy(),
                    }
                )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> dict:
        s = self.samples[idx]
        return {
            "x": torch.from_numpy(s["x"]),
            "y_soc": torch.from_numpy(s["y_soc"]),
            "y_soh": torch.from_numpy(s["y_soh"]),
        }

In [ ]:
# Model — FEM, Evaluator, Predictor, Mamba, LSTM, MDEN
from  torch.nn import RMSNorm
class FeatureExtractionModule(nn.Module):
    def __init__(
        self,
        channels: int = 5,
        in_length: int = SEQ_LEN,
        k_top_freq: int = K_TOP_FREQ,
        channel_expand: int = 64,
        groups: int = 8,
        num_rounds: int = 4,
    ):
        super().__init__()
        self.channels = channels
        self.in_length = in_length
        self.k_top_freq = k_top_freq
        self.num_rounds = num_rounds

        g = min(groups, channel_expand)
        while channel_expand % g != 0:
            g -= 1
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channel_expand, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                channel_expand,
                channel_expand,
                kernel_size=3,
                padding=1,
                groups=g,
            ),
            nn.ReLU(inplace=True),
            nn.Conv2d(channel_expand, channels, kernel_size=1),
        )

    def _select_periods(self, x: torch.Tensor) -> List[int]:
        B, C, L = x.shape
        dyn = x[:, 2:, :] if C >= 5 else x  # V, I, T
        amp = torch.abs(torch.fft.rfft(dyn, dim=-1))
        amp[..., 0] = 0.0
        amp = amp / (amp.mean(dim=-1, keepdim=True) + 1e-8)
        avg = amp.mean(dim=1)  # [B, F]
        n_freq = avg.size(-1)
        k = min(self.k_top_freq, max(n_freq - 1, 1))
        _, top_idx = torch.topk(avg, k=k, dim=-1)
        top_k_freq: List[int] = []
        for j in range(k):
            top_k_freq.append(int(torch.mode(top_idx[:, j], dim=0).values.item()) + 1)
        top_k_freq = sorted({max(1, f) for f in top_k_freq})
        while len(top_k_freq) < k:
            top_k_freq.append(min(top_k_freq[-1] + 1, max(n_freq - 1, 1)))
        return top_k_freq[:k]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, L = x.shape
        top_k_freq = self._select_periods(x)

        outs = []
        for f_j in top_k_freq:
            p_j = int(np.ceil(L / f_j))
            target = p_j * f_j
            if L < target:
                x_pad = F.pad(x, (0, target - L))
            else:
                x_pad = x[:, :, :target]

            x2d = x_pad.reshape(B, C, p_j, f_j)
            xc = x2d
            for _ in range(self.num_rounds):
                xc = self.conv(xc)

            weights = torch.softmax(x2d.reshape(B, C, -1), dim=-1).reshape_as(x2d)
            x_w = xc * weights
            rem = x_w.reshape(B, C, -1)[:, :, :L]
            outs.append(rem)

        return torch.stack(outs, dim=0).sum(dim=0)


class ExpertStack(nn.Module):

    def __init__(self, num_layers: int, channels: int = 5, in_length: int = SEQ_LEN):
        super().__init__()
        self.layers = nn.ModuleList(
            [FeatureExtractionModule(channels=channels, in_length=in_length) for _ in range(num_layers)]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = x
        for layer in self.layers:
            h = layer(h)
        return h


class Evaluator(nn.Module):

    def __init__(self, in_features: int = 5):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.net = nn.Sequential(
            nn.Linear(in_features, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, 2),
            nn.Softmax(dim=-1),
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # x: [B, C, L]
        pooled = self.pool(x).squeeze(-1)
        w = self.net(pooled)
        return w[:, 0], w[:, 1]


class SigmaPredictor(nn.Module):

    def __init__(self, in_channels: int = 5):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(in_channels, 1)
        self.softplus = nn.Softplus()

    def forward(self, f_w: torch.Tensor) -> torch.Tensor:
        # f_w: [B, C, L]
        z = self.pool(f_w).squeeze(-1)
        return self.softplus(self.fc(z)) + 1e-3


class SimplifiedMamba(nn.Module):

    def __init__(self, d_model: int = 32, d_state: int = 16, d_conv: int = 4, expand: int = 2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        inner = d_model * expand
        self.inner = inner

        self.in_norm = RMSNorm(d_model)
        self.in_proj_x = nn.Linear(d_model, inner, bias=False)
        self.in_proj_z = nn.Linear(d_model, inner, bias=False)
        self.conv1d = nn.Conv1d(
            inner, inner, kernel_size=d_conv, padding=d_conv - 1, groups=inner
        )
        self.x_proj = nn.Linear(inner, d_state * 2 + 1, bias=False)
        self.dt_proj = nn.Linear(1, inner, bias=True)

        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(1).repeat(1, inner)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(inner))
        self.out_proj = nn.Linear(inner, d_model, bias=False)
        self.out_norm = RMSNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, D]
        B, L, _ = x.shape

        n = self.in_norm(x)
        x_b = self.in_proj_x(n)
        z = self.in_proj_z(n)

        x_b = x_b.transpose(1, 2)
        x_b = self.conv1d(x_b)[:, :, :L]
        x_b = x_b.transpose(1, 2)
        x_b = F.silu(x_b)

        x_dbl = self.x_proj(x_b)
        dt, B_par, C_par = torch.split(
            x_dbl, [1, self.d_state, self.d_state], dim=-1
        )
        dt = F.softplus(self.dt_proj(dt))
        A = -torch.exp(self.A_log)

        y = self._scan(x_b, dt, A, B_par, C_par)
        y = y + x_b * self.D
        y = y * F.silu(z)
        y = self.out_proj(y)
        return self.out_norm(y)

    def _scan(
        self,
        u: torch.Tensor,
        dt: torch.Tensor,
        A: torch.Tensor,
        B: torch.Tensor,
        C: torch.Tensor,
    ) -> torch.Tensor:
        Bsz, L, inner = u.shape
        N = A.size(0)
        h = u.new_zeros(Bsz, N, inner)
        ys = []
        for t in range(L):
            dt_t = dt[:, t].unsqueeze(1)
            dA = torch.exp(A.unsqueeze(0) * dt_t)
            dB = dt_t * B[:, t].unsqueeze(-1)
            h = dA * h + dB * u[:, t].unsqueeze(1)
            y_t = (h * C[:, t].unsqueeze(-1)).sum(dim=1)
            ys.append(y_t)
        return torch.stack(ys, dim=1)


class PrePredictor(nn.Module):

    def __init__(self, in_channels: int = 2, hidden: int = 64, horizon: int = PRED_HORIZON):
        super().__init__()
        self.horizon = horizon
        self.conv1 = nn.Conv1d(in_channels, hidden, kernel_size=1)
        self.conv2 = nn.Conv1d(hidden, 2 * horizon, kernel_size=1)
        with torch.no_grad():
            self.conv2.bias.zero_()
            self.conv2.bias[horizon:].fill_(2.2)

    def forward(self, f_f: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        out = F.relu(self.conv1(f_f))
        out = self.conv2(out)
        out = out.mean(dim=-1)
        soc = out[:, : self.horizon]
        soh = out[:, self.horizon :]
        return soc, soh


class MDEN(nn.Module):

    def __init__(
        self,
        input_length: int = SEQ_LEN,
        input_features: int = 5,
        hidden_size: int = 32,
        pred_horizon: int = PRED_HORIZON,
    ):
        super().__init__()
        self.input_length = input_length
        self.hidden_size = hidden_size

        self.shared = ExpertStack(4, channels=input_features, in_length=input_length)
        self.soc_specific = ExpertStack(2, channels=input_features, in_length=input_length)
        self.soh_specific = ExpertStack(6, channels=input_features, in_length=input_length)

        self.soc_evaluator = Evaluator(input_features)
        self.soh_evaluator = Evaluator(input_features)

        self.soc_input_proj = nn.Linear(input_features, hidden_size)
        self.soh_input_proj = nn.Linear(input_features, hidden_size)

        self.soc_lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            batch_first=True,
        )
        self.soh_mamba = SimplifiedMamba(d_model=hidden_size, d_state=16, d_conv=4, expand=2)

        self.soc_head = nn.Linear(hidden_size, pred_horizon)
        self.soh_head = nn.Linear(hidden_size, pred_horizon)
        with torch.no_grad():
            self.soh_head.bias.fill_(2.2)

        self.pre = PrePredictor(in_channels=2, hidden=128, horizon=pred_horizon)
        self.sigma_soc = SigmaPredictor(input_features)
        self.sigma_soh = SigmaPredictor(input_features)
        self.head_mix = nn.Parameter(torch.tensor(0.5))

    def forward(self, x: torch.Tensor) -> dict:
        sf = self.shared(x)
        tsf_soc = self.soc_specific(x)
        tsf_soh = self.soh_specific(x)

        w_soc1, w_soc2 = self.soc_evaluator(x)
        w_soh1, w_soh2 = self.soh_evaluator(x)

        fw_soc = w_soc1.view(-1, 1, 1) * sf + w_soc2.view(-1, 1, 1) * tsf_soc
        fw_soh = w_soh1.view(-1, 1, 1) * sf + w_soh2.view(-1, 1, 1) * tsf_soh

        soc_seq = self.soc_input_proj(fw_soc.transpose(1, 2))  # [B, L, H]
        soh_seq = self.soh_input_proj(fw_soh.transpose(1, 2))

        soc_out, _ = self.soc_lstm(soc_seq)
        f_soc = soc_out[:, -1, :]

        soh_out = self.soh_mamba(soh_seq)
        f_soh = soh_out[:, -1, :]

        f_f = torch.stack([f_soc, f_soh], dim=1)
        soc_pre, soh_pre = self.pre(f_f)
        soc_dir = self.soc_head(f_soc)
        soh_dir = self.soh_head(f_soh)
        mix = torch.sigmoid(self.head_mix)
        soc_pred = torch.sigmoid(mix * soc_pre + (1.0 - mix) * soc_dir)
        soh_pred = torch.sigmoid(mix * soh_pre + (1.0 - mix) * soh_dir)

        return {
            "soc": soc_pred,
            "soh": soh_pred,
            "sigma_soc": self.sigma_soc(fw_soc),
            "sigma_soh": self.sigma_soh(fw_soh),
            "fw_soc": fw_soc,
            "fw_soh": fw_soh,
        }


class JointUncertaintyLoss(nn.Module):

    def __init__(self, sigma_max: float = 2.0):
        super().__init__()
        self.sigma_max = sigma_max

    def forward(
        self,
        pred_soc: torch.Tensor,
        pred_soh: torch.Tensor,
        target_soc: torch.Tensor,
        target_soh: torch.Tensor,
        sigma_soc: torch.Tensor,
        sigma_soh: torch.Tensor,
    ) -> Tuple[torch.Tensor, dict]:
        mse_soc = F.mse_loss(pred_soc, target_soc)
        mse_soh = F.mse_loss(pred_soh, target_soh)

        s_soc = sigma_soc.mean().clamp(1e-3, self.sigma_max)
        s_soh = sigma_soh.mean().clamp(1e-3, self.sigma_max)

        loss = (
            (1.0 / (2.0 * s_soc ** 2)) * mse_soc
            + (1.0 / (2.0 * s_soh ** 2)) * mse_soh
            + torch.log(s_soc + 1.0)
            + torch.log(s_soh + 1.0)
        )
        stats = {
            "mse_soc": float(mse_soc.detach()),
            "mse_soh": float(mse_soh.detach()),
            "sigma_soc": float(s_soc.detach()),
            "sigma_soh": float(s_soh.detach()),
            "pred_soh_mean": float(pred_soh.detach().mean()),
            "tgt_soh_mean": float(target_soh.detach().mean()),
        }
        return loss, stats

In [ ]:
# Metrics & training
@torch.no_grad()
def rmse(pred: torch.Tensor, target: torch.Tensor) -> float:
    return float(torch.sqrt(torch.mean((pred - target) ** 2)).cpu())


@torch.no_grad()
def mae(pred: torch.Tensor, target: torch.Tensor) -> float:
    return float(torch.mean(torch.abs(pred - target)).cpu())


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: JointUncertaintyLoss,
    optimizer: Optional[torch.optim.Optimizer],
    device: torch.device,
) -> dict:
    train = optimizer is not None
    model.train(train)

    total_loss = 0.0
    soc_preds, soc_tgts = [], []
    soh_preds, soh_tgts = [], []
    sum_stats = {
        "mse_soc": 0.0,
        "mse_soh": 0.0,
        "sigma_soc": 0.0,
        "sigma_soh": 0.0,
        "pred_soh_mean": 0.0,
        "tgt_soh_mean": 0.0,
    }

    for batch in loader:
        x = batch["x"].to(device)
        y_soc = batch["y_soc"].to(device)
        y_soh = batch["y_soh"].to(device)

        with torch.set_grad_enabled(train):
            out = model(x)
            loss, stats = criterion(
                out["soc"], out["soh"], y_soc, y_soh, out["sigma_soc"], out["sigma_soh"]
            )
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        total_loss += float(loss.detach())
        for k in sum_stats:
            sum_stats[k] += stats[k]
        soc_preds.append(out["soc"].detach().cpu())
        soc_tgts.append(y_soc.detach().cpu())
        soh_preds.append(out["soh"].detach().cpu())
        soh_tgts.append(y_soh.detach().cpu())

    n = max(len(loader), 1)
    sp = torch.cat(soc_preds)
    st = torch.cat(soc_tgts)
    hp = torch.cat(soh_preds)
    ht = torch.cat(soh_tgts)
    return {
        "loss": total_loss / n,
        "rmse_soc": rmse(sp, st),
        "rmse_soh": rmse(hp, ht),
        "mae_soc": mae(sp, st),
        "mae_soh": mae(hp, ht),
        **{k: v / n for k, v in sum_stats.items()},
    }


In [ ]:
"""def main():
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    bat_dict = load_bat_dict(DATA_DIR)
    series = build_cycle_series(bat_dict)

    present = {s["bat_id"] for s in series}
    train_cells = [c for c in TRAIN_CELLS if c in present]
    test_cells = [c for c in TEST_CELLS if c in present]
    if len(train_cells) < 2:
        all_cells = sorted(present)
        test_cells = all_cells[-2:]
        train_cells = all_cells[:-2]
        print(f"Fallback split — train: {len(train_cells)}, test: {test_cells}")
    else:
        print(f"Train cells: {train_cells}")
        print(f"Test cells:  {test_cells}")

    train_series, feat_stats = normalize_series(
        [s for s in series if s["bat_id"] in set(train_cells)], stats=None
    )
    test_series, _ = normalize_series(
        [s for s in series if s["bat_id"] in set(test_cells)], stats=feat_stats
    )
    train_full = SOCSOHDataset(train_series, cell_ids=None)
    test_ds = SOCSOHDataset(test_series, cell_ids=None)

    n_train = int(0.8 * len(train_full))
    n_val = len(train_full) - n_train
    train_ds, val_ds = torch.utils.data.random_split(
        train_full, [n_train, n_val], generator=torch.Generator().manual_seed(SEED)
    )

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    print(f"Samples — train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}")

    y0 = train_full[0]["y_soh"].numpy()
    ys = np.stack([train_full[i]["y_soh"].numpy() for i in range(0, len(train_full), max(len(train_full) // 200, 1))])
    print(f"Train y_soh sample mean={ys.mean():.3f} std={ys.std():.3f} (first window {y0})")

    model = MDEN().to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {n_params:,}")

    criterion = JointUncertaintyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    history = {"train": [], "val": []}
    best_val = float("inf")
    ckpt_path = os.path.join(DATA_DIR, "mden_weights.pth")

    for epoch in range(1, EPOCHS + 1):
        tr = run_epoch(model, train_loader, criterion, optimizer, device)
        va = run_epoch(model, val_loader, criterion, None, device)
        history["train"].append(tr)
        history["val"].append(va)

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"Train loss {tr['loss']:.4f} (SOC RMSE {tr['rmse_soc']:.4f}, SOH RMSE {tr['rmse_soh']:.4f}) | "
            f"Val loss {va['loss']:.4f} (SOC RMSE {va['rmse_soc']:.4f}, SOH RMSE {va['rmse_soh']:.4f}) | "
            f"predSOH {va['pred_soh_mean']:.3f} vs tgt {va['tgt_soh_mean']:.3f}"
        )

        if va["rmse_soh"] + va["rmse_soc"] < best_val:
            best_val = va["rmse_soh"] + va["rmse_soc"]
            torch.save(model.state_dict(), ckpt_path)
            print(f"  ↑ saved best → {ckpt_path}")

    if os.path.isfile(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
    te = run_epoch(model, test_loader, criterion, None, device)
    print(
        f"\nTest | loss {te['loss']:.4f} | "
        f"SOC RMSE {te['rmse_soc']:.4f} MAE {te['mae_soc']:.4f} | "
        f"SOH RMSE {te['rmse_soh']:.4f} MAE {te['mae_soh']:.4f}"
    )

    # Curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot([h["loss"] for h in history["train"]], label="Train")
    axes[0].plot([h["loss"] for h in history["val"]], label="Val")
    axes[0].set_title("Joint loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot([h["rmse_soc"] for h in history["val"]], label="Val SOC RMSE")
    axes[1].plot([h["rmse_soh"] for h in history["val"]], label="Val SOH RMSE")
    axes[1].set_title("Validation RMSE")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    main()"""

'def main():\n    set_seed(SEED)\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    print(f"Device: {device}")\n\n    bat_dict = load_bat_dict(DATA_DIR)\n    series = build_cycle_series(bat_dict)\n\n    present = {s["bat_id"] for s in series}\n    train_cells = [c for c in TRAIN_CELLS if c in present]\n    test_cells = [c for c in TEST_CELLS if c in present]\n    if len(train_cells) < 2:\n        all_cells = sorted(present)\n        test_cells = all_cells[-2:]\n        train_cells = all_cells[:-2]\n        print(f"Fallback split — train: {len(train_cells)}, test: {test_cells}")\n    else:\n        print(f"Train cells: {train_cells}")\n        print(f"Test cells:  {test_cells}")\n\n    train_series, feat_stats = normalize_series(\n        [s for s in series if s["bat_id"] in set(train_cells)], stats=None\n    )\n    test_series, _ = normalize_series(\n        [s for s in series if s["bat_id"] in set(test_cells)], stats=feat_stats\n    )\n    train_full = SOC

In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader


@torch.no_grad()
def compute_all_metrics(pred: np.ndarray, target: np.ndarray) -> dict:
    pred = pred.reshape(-1); target = target.reshape(-1)
    eps = 1e-8
    return {
        "RMSE":  float(np.sqrt(np.mean((pred - target) ** 2))),
        "R2":    1.0 - float(np.sum((target - pred) ** 2)) /
                       (float(np.sum((target - target.mean()) ** 2)) + eps),
        "WMAPE": float(np.sum(np.abs(target - pred)) /
                       (np.sum(np.abs(target)) + eps)),
        "MAE":   float(np.mean(np.abs(pred - target))),
        "MdAE":  float(np.median(np.abs(pred - target))),
    }


@torch.no_grad()
def evaluate_test(model, loader, device):
    model.eval()
    sp, st, hp, ht = [], [], [], []
    for batch in loader:
        x = batch["x"].to(device)
        y_soc = batch["y_soc"].to(device)
        y_soh = batch["y_soh"].to(device)
        out = model(x)
        sp.append(out["soc"].detach().cpu().numpy())
        st.append(y_soc.detach().cpu().numpy())
        hp.append(out["soh"].detach().cpu().numpy())
        ht.append(y_soh.detach().cpu().numpy())
    sp = np.concatenate(sp); st = np.concatenate(st)
    hp = np.concatenate(hp); ht = np.concatenate(ht)
    return {
        "SOC": compute_all_metrics(sp, st),
        "SOH": compute_all_metrics(hp, ht),
    }


present = {s["bat_id"] for s in series}
train_cells = [c for c in TRAIN_CELLS if c in present]
test_cells  = [c for c in TEST_CELLS  if c in present]

train_series, feat_stats = normalize_series(
    [s for s in series if s["bat_id"] in set(train_cells)], stats=None
)
test_series, _ = normalize_series(
    [s for s in series if s["bat_id"] in set(test_cells)], stats=feat_stats
)

print(f"Train cells: {train_cells}")
print(f"Test  cells: {test_cells}")
print("Feature stats (from train):")
for k, v in feat_stats.items():
    print(f"  {k:6s} mean={v['mean']:.4f}  std={v['std']:.4f}")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt_path = os.path.join(DATA_DIR, "mden_weights.pth")
assert os.path.isfile(ckpt_path), f"Нет файла весов: {ckpt_path}"

model = MDEN().to(device)
model.load_state_dict(torch.load(ckpt_path, map_location=device))
print(f"Loaded weights: {ckpt_path}")


results = {}
for cell in test_cells:
    ds_cell = SOCSOHDataset(
        [s for s in test_series if s["bat_id"] == cell],
        cell_ids=None,
    )
    loader = DataLoader(ds_cell, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS)
    results[cell] = evaluate_test(model, loader, device)


print("\n" + "=" * 90)
print(f"{'Battery':<8} {'Task':<5} {'RMSE':>8} {'R2':>8} {'WMAPE':>8} {'MAE':>8} {'MdAE':>8}")
print("-" * 90)
for cell, m in results.items():
    for task in ("SOC", "SOH"):
        mm = m[task]
        print(f"{cell:<8} {task:<5} {mm['RMSE']:>8.4f} {mm['R2']:>8.4f} "
              f"{mm['WMAPE']:>8.4f} {mm['MAE']:>8.4f} {mm['MdAE']:>8.4f}")
    print("-" * 90)

Feature norm (train, reused for val/test):
  cycle  mean=366.8975  std=226.7618
  time   mean=30.3262  std=240.9439
  V      mean=3.4001  std=0.3430
  I      mean=2.3752  std=2.2284
  T      mean=33.2495  std=2.2286
Train cells: ['b1c14', 'b1c15', 'b1c16', 'b1c17', 'b1c18', 'b1c19', 'b1c20', 'b1c21']
Test  cells: ['b1c12', 'b1c13']
Feature stats (from train):
  cycle  mean=366.8975  std=226.7618
  time   mean=30.3262  std=240.9439
  V      mean=3.4001  std=0.3430
  I      mean=2.3752  std=2.2284
  T      mean=33.2495  std=2.2286
Loaded weights: /content/drive/MyDrive/mden_weights.pth

Battery  Task      RMSE       R2    WMAPE      MAE     MdAE
------------------------------------------------------------------------------------------
b1c12    SOC     0.0174   0.9972   0.0130   0.0081   0.0025
b1c12    SOH     0.0172   0.7587   0.0111   0.0106   0.0050
------------------------------------------------------------------------------------------
b1c13    SOC     0.0186   0.9968   0.0153   0.